# CRUSADER — F03 SIGISMUND (GitHub Actions)
## Rendu Remotion distribue → short_render.mp4

> *"Sigismund stood unmoved, and from him radiated the Emperor's will."*

---

**Ce notebook utilise GitHub Actions pour le rendu distribue (10 workers gratuits).**  
**Temps estime : ~5-10 min pour une video courte. Aucune CB requise.**

### Etapes :
1. Montage Google Drive
2. Installation dependances (requests)
3. Authentification GitHub (GITHUB_TOKEN)
4. Telechargement des scripts depuis GitHub
5. Configuration des chemins
6. Validation CUSTOS check-out
7. Upload assets vers GitHub Release (temporaire)
8b. Trigger GitHub Actions (10 workers)
8c. Polling du statut du rendu
9. Telechargement de la video finale
10. Sauvegarde sur Drive
11. Validation CUSTOS check-in
12. Apercu et telechargement


---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Etape 2 — Installation dependances


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', '-q'], check=True)
import requests
print(f'[OK] requests {requests.__version__} installe')


---
## Etape 3 — Authentification GitHub

> Creez un **Personal Access Token** sur **https://github.com/settings/tokens**  
> Permissions requises : `repo` (full) + `workflow`
>
> Ajoutez-le dans l'onglet Secrets Colab (icone cle a gauche) :
> nom : `GITHUB_TOKEN`, valeur : votre token. Activer "Notebook access".


In [ ]:
import os
from google.colab import userdata

# ── GITHUB_TOKEN depuis Colab Secrets ─────────────────────────────────────
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')

GH_REPO = 'kioka8877-ux/CRUSADER'

print('[OK] GITHUB_TOKEN charge depuis Colab Secrets')
print(f'[OK] Repo cible : {GH_REPO}')


---
## Etape 4 — Telechargement des scripts depuis GitHub


In [ ]:
import urllib.request, os

REPO_RAW    = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
SCRIPTS_DIR = '/content/crusader_scripts'
os.makedirs(SCRIPTS_DIR, exist_ok=True)

files_to_download = [
    ('F03_SIGISMUND/CODEBASE/crs_f03_gh_trigger.py', 'crs_f03_gh_trigger.py'),
    ('CRS_CUSTOS.py',                                 'CRS_CUSTOS.py'),
]

for rel_path, dest_name in files_to_download:
    dest_path = os.path.join(SCRIPTS_DIR, dest_name)
    urllib.request.urlretrieve(f'{REPO_RAW}/{rel_path}', dest_path)
    print(f'[OK] {dest_name}')

print('\nScripts telecharges.')


---
## Etape 5 — Configuration des chemins

> **Modifiez `DRIVE_BASE` et `COMPOSITION` si necessaire.**


In [ ]:
import os

# ── MODIFIEZ ICI SI NECESSAIRE ───────────────────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/DRIVE_CRUSADER'
COMPOSITION = 'CrusaderShort'
# ─────────────────────────────────────────────────────────────────────────────

SCRIPTS_DIR = '/content/crusader_scripts'
F03_IN      = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'IN')
F03_OUT     = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'OUT')
os.makedirs(F03_OUT, exist_ok=True)

print('Configuration :')
print(f'  F03 IN   : {F03_IN}')
print(f'  F03 OUT  : {F03_OUT}')
print(f'  Workers  : 10 (fixe dans le workflow GitHub Actions)')
print()

checks = {
    'timing.json':     os.path.isfile(os.path.join(F03_IN, 'timing.json')),
    'roadmap.json':    os.path.isfile(os.path.join(F03_IN, 'roadmap.json')),
    'audio_clean.mp3': os.path.isfile(os.path.join(F03_IN, 'audio_clean.mp3')),
    'images/':         os.path.isdir(os.path.join(F03_IN, 'images')),
}
all_ok = True
for name, ok in checks.items():
    status = 'OK' if ok else 'MANQUANT'
    print(f'  {name}: {status}')
    if not ok:
        all_ok = False

if not all_ok:
    print('\n[STOP] Fichiers manquants dans F03/IN/. Corrigez avant de continuer.')
else:
    print('\n[OK] Tous les assets presents.')


---
## Étape 6 — Validation CUSTOS check-out (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(SCRIPTS_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-out FAIL. Vérifiez les fichiers dans F03/IN/.')

---
## Etape 7 — Upload assets vers GitHub Release (temporaire)

> Cree une Release GitHub temporaire avec un tag unique (run_id).  
> Upload les fichiers : `timing.json`, `roadmap.json`, `audio_clean.mp3`, `images.zip`  
> Les workers GitHub Actions telechargeront ces assets depuis la Release.


In [ ]:
import os, sys, time
sys.path.insert(0, SCRIPTS_DIR)
from crs_f03_gh_trigger import upload_assets_to_release

# ── Generer un run_id unique ──────────────────────────────────────────────
RUN_ID = f'f03-{int(time.time())}'
print(f'[UPLOAD] run_id : {RUN_ID}')

# ── Upload des assets vers GitHub Release ────────────────────────────────
release_url = upload_assets_to_release(
    f03_in=F03_IN,
    run_id=RUN_ID,
    github_token=os.environ['GITHUB_TOKEN'],
    repo=GH_REPO,
)
print(f'[OK] Assets disponibles sur : {release_url}')


---
## Etape 8b — Trigger GitHub Actions (10 workers)

> **Lance 10 workers en parallele sur GitHub Actions (gratuit, sans CB).**  
> Le workflow lit les assets depuis la Release temporaire.  
> **Une fois lance, tu peux fermer Colab — les workers sont independants.**


In [ ]:
import os, sys, json
sys.path.insert(0, SCRIPTS_DIR)
from crs_f03_gh_trigger import trigger_workflow

# ── Lecture du nombre total de frames ────────────────────────────────────
with open(os.path.join(F03_IN, 'roadmap.json')) as f:
    roadmap = json.load(f)

total_frames = max(scene['end_frame'] for scene in roadmap['timeline']) + 1
fps          = roadmap['meta']['fps']
print(f'[INFO] Total frames : {total_frames} ({total_frames / fps:.1f} sec @ {fps} fps)')

# ── Trigger GitHub Actions ────────────────────────────────────────────────
GH_RUN_ID = trigger_workflow(
    run_id=RUN_ID,
    fps=fps,
    composition=COMPOSITION,
    total_frames=total_frames,
    github_token=os.environ['GITHUB_TOKEN'],
    repo=GH_REPO,
)
print()
print('[OK] 10 workers GitHub Actions lances — independants de Colab.')
print(f'[INFO] Tu peux fermer Colab. Reviens dans ~5-10 min et lance Etape 8c.')
print(f'[INFO] Suivi : https://github.com/{GH_REPO}/actions/runs/{GH_RUN_ID}')


---
## Etape 8c — Polling du statut GitHub Actions

> Surveille le run en cours et affiche la progression toutes les 15 secondes.  
> **Si Colab a crashe pendant l'Etape 8b :** relance etapes 1-3, redefinis `GH_RUN_ID` manuellement  
> depuis l'URL GitHub Actions, puis relance cette cellule.


In [ ]:
import os, sys
sys.path.insert(0, SCRIPTS_DIR)
from crs_f03_gh_trigger import poll_run_status

print(f'[POLL] Surveillance du run GitHub Actions {GH_RUN_ID}...')
print(f'[INFO] 10 workers en parallele — relance depuis Etape 1-3 si Colab a crashe.\n')

status = poll_run_status(
    gh_run_id=GH_RUN_ID,
    github_token=os.environ['GITHUB_TOKEN'],
    repo=GH_REPO,
)
print(f'\n[OK] Rendu termine — statut : {status}')
print('[INFO] Lance maintenant Etape 9 (telechargement).')


---
## Etape 9 — Telechargement de la video finale

> Telecharge l'artifact `resultat-final` depuis GitHub Actions.  
> Extrait `short_render.mp4` dans le dossier OUT/ de Drive.  
> Supprime automatiquement la Release temporaire apres telechargement.


In [ ]:
import os, sys
sys.path.insert(0, SCRIPTS_DIR)
from crs_f03_gh_trigger import download_final_artifact

output_path = download_final_artifact(
    gh_run_id=GH_RUN_ID,
    run_id=RUN_ID,
    github_token=os.environ['GITHUB_TOKEN'],
    repo=GH_REPO,
    output_dir=F03_OUT,
)

size_mb = os.path.getsize(output_path) / 1024 / 1024
final_video_bytes = open(output_path, 'rb').read()
print(f'[OK] short_render.mp4 — {size_mb:.1f} MB')


---
## Étape 10 — Sauvegarde sur Drive (F03/OUT/)

In [ ]:
import os

output_path = os.path.join(F03_OUT, 'short_render.mp4')
os.makedirs(F03_OUT, exist_ok=True)

with open(output_path, 'wb') as f:
    f.write(final_video_bytes)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'[OK] Vidéo sauvegardée → {output_path}')
print(f'     Taille : {size_mb:.1f} MB')

---
## Étape 11 — Validation CUSTOS check-in (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(SCRIPTS_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-in FAIL. short_render.mp4 absent ou trop petit.')
else:
    print('[OK] short_render.mp4 validé — prêt pour transfert vers F04.')

---
## Étape 12 — Aperçu et téléchargement

In [ ]:
import os
from IPython.display import Video, display

output_path = os.path.join(F03_OUT, 'short_render.mp4')

if os.path.isfile(output_path):
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f'short_render.mp4 — {size_mb:.1f} MB')
    print(f'Chemin : {output_path}')
    print()
    display(Video(output_path, embed=True, width=360))
else:
    print('[ERREUR] short_render.mp4 introuvable.')

In [ ]:
# Téléchargement direct depuis Colab (optionnel)
from google.colab import files
files.download(output_path)